# 04 Neural models in Keras: MLP, CNN, BiLSTM

Notebook 03 showed that TF-IDF + logistic regression already reaches **99.0%** on Mistral but only **82.8%** on the
unseen Qwen (it finds just 67% of Qwen reviews). The question here: **do neural models generalise better to an unseen
LLM, and at what false-positive cost?**

| Section | Model | Input |
| --- | --- | --- |
| 2 | MLP (Dense) | TF-IDF of words and word pairs, **3 preprocessing variants (ablation)** |
| 3 | CNN (Conv1D) | word sequence → embedding |
| 4 | BiLSTM | word sequence → embedding |

**Architectures from Kryštof's SST-2 notebook** (Neural Networks course, 4IZ175): the same MLP, CNN and BiLSTM
baselines, applied to our task. What we changed and why:

| Change | Why |
| --- | --- |
| Sequence length 200 instead of 50 | our reviews are longer (median ~20 words, up to 300) than SST-2 sentences (median 7) |
| MLP preprocessing is tested, not fixed | his MLP lowercases, removes punctuation and stopwords; notebook 03 showed that case and punctuation are our strongest signals → ablation in section 2 |
| `mask_zero=True` in the BiLSTM | most reviews are much shorter than 200 tokens; without the mask the LSTM reads long runs of padding |
| No Hyperband tuning | in his notebook tuning on the validation set did not help (MLP −3.4 points, CNN −0.2); here the models already reach ~99% on Mistral, and the real question (Qwen) cannot be tuned on without leaking the held-out test |

Every model is trained on `train`, stopped early on `val`, and scored on `val`, `test`, `heldout_qwen` and
`reserve_human`. Metrics go to `TA2/results/04_models/metrics.csv`, every run to `runs_log.csv`.

**How to run:** Runtime → Change runtime type → **T4 or L4 GPU** → Run all (about 10–15 minutes).
Neural networks are not exactly reproducible on a GPU: a second run can differ by a few tenths of a percent.

## 1. Setup

In [ ]:
import json, time, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from google.colab import drive
drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/TA2/data/final/dataset_v1.csv")
RESULTS = Path("/content/drive/MyDrive/TA2/results/04_models")      # metrics and run log (no review text)
PREDS = RESULTS / "predictions"                                      # p(AI) per review_id, for the error analysis
PREDS.mkdir(parents=True, exist_ok=True)
SEED = 42

df = pd.read_csv(DATA)
df["text"] = df.text.astype(str)
SPLITS = ["train", "val", "test", "heldout_qwen", "reserve_human"]
splits = {s: df[df.split == s].reset_index(drop=True) for s in SPLITS}
print({s: len(d) for s, d in splits.items()})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 12380, 'val': 2652, 'test': 2654, 'heldout_qwen': 5516, 'reserve_human': 16090}


In [ ]:
def evaluate(name, probs, settings):
    """probs: dict split -> array of p(AI) for that split (same order as splits[split]).
    Scores every split, saves the predictions and appends the metrics to metrics.csv and runs_log.csv."""
    rows = []
    for s, p in probs.items():
        d = splits[s]; p = np.asarray(p).ravel(); pred = (p >= 0.5).astype(int)
        pd.DataFrame({"review_id": d.review_id, "split": s, "label": d.label, "p_ai": p}) \
          .to_csv(PREDS / f"{name}__{s}.csv", index=False)
        row = {"model": name, "split": s, "n": len(d)}
        if s == "reserve_human":        # only real players: every "AI" prediction is a false positive
            row["false_positive_rate"] = pred.mean()
        else:
            row.update(accuracy=accuracy_score(d.label, pred), precision_ai=precision_score(d.label, pred, zero_division=0),
                       recall_ai=recall_score(d.label, pred), f1_ai=f1_score(d.label, pred),
                       roc_auc=roc_auc_score(d.label, p), false_positive_rate=pred[d.label.values == 0].mean())
        rows.append(row)
    new = pd.DataFrame(rows)
    path = RESULTS / "metrics.csv"
    old = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(old):
        old = old[old.model != name]                 # re-running a model replaces its rows
    pd.concat([old, new]).round(4).to_csv(path, index=False)
    # run log: every training run with its settings, for the failure journal
    log = {"date": datetime.datetime.now().isoformat(timespec="minutes"), "model": name, "settings": json.dumps(settings),
           **{f"acc_{r['split']}": round(r.get("accuracy", np.nan), 4) for r in rows},
           "fpr_reserve": round(new[new.split == "reserve_human"].false_positive_rate.iloc[0], 4)}
    pd.DataFrame([log]).to_csv(RESULTS / "runs_log.csv", mode="a", header=not (RESULTS / "runs_log.csv").exists(), index=False)
    return new.set_index("split")[["accuracy", "precision_ai", "recall_ai", "f1_ai", "roc_auc", "false_positive_rate"]].round(3)

In [ ]:
import tensorflow as tf
import keras
from keras import layers
keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

MAX_TOKENS = 20000   # vocabulary size (most frequent words / word pairs in train)
SEQ_LEN = 200        # CNN / BiLSTM: reviews are cut or padded to 200 tokens (we kept reviews up to 300 words;
                     # 95% of them are shorter than 200)
BATCH = 64

def to_dataset(d, with_labels=True):
    texts = tf.constant(d.text.values)
    if with_labels:
        ds = tf.data.Dataset.from_tensor_slices((texts, d.label.values.astype("float32")))
        if (d.split == "train").all():
            # shuffle the training data (in the file all human reviews come first, then all AI ones);
            # without this the network sees only one class per part of the epoch and learns nothing
            ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
        return ds.batch(BATCH)
    return tf.data.Dataset.from_tensor_slices(texts).batch(BATCH)

def predict_all(model, vectorize):
    """p(AI) for every evaluation split."""
    return {s: model.predict(to_dataset(splits[s], with_labels=False).map(vectorize), verbose=0)
            for s in ["val", "test", "heldout_qwen", "reserve_human"]}

early_stop = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)  # as in Kryštof's notebook

TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. MLP on TF-IDF + ablation: how should the text be prepared?

Keras' default (`lower_and_strip_punctuation`) lowercases the text and removes punctuation. But notebook 03 showed that
**case and punctuation are among the strongest signals** (humans start lowercase, write "i", skip the final full stop).
So we compare three preprocessing variants with the same MLP:

| Variant | What it does |
| --- | --- |
| `default` | lowercase + remove punctuation (Keras default) |
| `keep_case_punct` | keep capital letters, keep `! ? . , … #` as separate tokens |
| `default_no_stopwords` | default + remove English stopwords (the, a, is, …) |

The MLP is Kryštof's baseline: TF-IDF of words and word pairs (20,000) → Dense(64) → Dropout(0.5) → Dense(32) → Dropout(0.5) → sigmoid, optimizer AdamW.

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

PUNCT = r"([!?.,…#])"
STOPWORDS = r"\b(" + "|".join(sorted(ENGLISH_STOP_WORDS)) + r")\b"

def keep_case_punct(text):
    # put spaces around punctuation so that "great!" becomes the tokens "great" and "!"
    return tf.strings.regex_replace(text, PUNCT, r" \1 ")

def default_no_stopwords(text):
    text = tf.strings.lower(text)
    text = tf.strings.regex_replace(text, r"[^\w\s']", " ")
    return tf.strings.regex_replace(text, STOPWORDS, " ")

STANDARDIZE = {"default": "lower_and_strip_punctuation", "keep_case_punct": keep_case_punct,
               "default_no_stopwords": default_no_stopwords}

train_text = tf.data.Dataset.from_tensor_slices(splits["train"].text.values).batch(256)

def train_mlp(variant, epochs=20):
    vec = layers.TextVectorization(max_tokens=MAX_TOKENS, output_mode="tf_idf", ngrams=2,
                                   standardize=STANDARDIZE[variant])
    vec.adapt(train_text)                    # vocabulary and IDF from train only
    vectorize = lambda x, *y: (vec(x), *y)
    model = keras.Sequential([
        keras.Input(shape=(MAX_TOKENS,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adamw", loss="binary_crossentropy", metrics=["accuracy"])
    hist = model.fit(to_dataset(splits["train"]).map(vectorize), validation_data=to_dataset(splits["val"]).map(vectorize),
                     epochs=epochs, callbacks=[early_stop], verbose=2)
    settings = {"type": "MLP", "standardize": variant, "max_tokens": MAX_TOKENS, "ngrams": 2, "dense": [64, 32],
                "dropout": 0.5, "optimizer": "adamw", "epochs_run": len(hist.history["loss"])}
    return evaluate(f"MLP tfidf ({variant})", predict_all(model, lambda x: vec(x)), settings)

ablation = {}
for variant in STANDARDIZE:
    print(f"\n===== MLP, preprocessing: {variant} =====")
    ablation[variant] = train_mlp(variant)
    display(ablation[variant])


===== MLP, preprocessing: default =====
Epoch 1/20
194/194 - 13s - 69ms/step - accuracy: 0.9149 - loss: 0.2367 - val_accuracy: 0.9872 - val_loss: 0.0418
Epoch 2/20
194/194 - 1s - 5ms/step - accuracy: 0.9887 - loss: 0.0427 - val_accuracy: 0.9860 - val_loss: 0.0357
Epoch 3/20
194/194 - 1s - 5ms/step - accuracy: 0.9945 - loss: 0.0199 - val_accuracy: 0.9868 - val_loss: 0.0425
Epoch 4/20
194/194 - 1s - 5ms/step - accuracy: 0.9967 - loss: 0.0162 - val_accuracy: 0.9842 - val_loss: 0.0710


,accuracy,precision_ai,recall_ai,f1_ai,roc_auc,false_positive_rate
split,,,,,,
val,0.987,0.986,0.988,0.987,0.999,0.014
test,0.986,0.986,0.986,0.986,0.999,0.014
heldout_qwen,0.869,0.980,0.754,0.852,0.955,0.016
reserve_human,NaN,NaN,NaN,NaN,NaN,0.017



===== MLP, preprocessing: keep_case_punct =====
Epoch 1/20
194/194 - 7s - 34ms/step - accuracy: 0.9145 - loss: 0.2442 - val_accuracy: 0.9917 - val_loss: 0.0322
Epoch 2/20
194/194 - 1s - 5ms/step - accuracy: 0.9905 - loss: 0.0389 - val_accuracy: 0.9928 - val_loss: 0.0211
Epoch 3/20
194/194 - 1s - 5ms/step - accuracy: 0.9966 - loss: 0.0158 - val_accuracy: 0.9898 - val_loss: 0.0306
Epoch 4/20
194/194 - 1s - 5ms/step - accuracy: 0.9969 - loss: 0.0122 - val_accuracy: 0.9928 - val_loss: 0.0193
Epoch 5/20
194/194 - 1s - 5ms/step - accuracy: 0.9988 - loss: 0.0065 - val_accuracy: 0.9921 - val_loss: 0.0217


,accuracy,precision_ai,recall_ai,f1_ai,roc_auc,false_positive_rate
split,,,,,,
val,0.993,0.993,0.992,0.993,1.000,0.007
test,0.993,0.994,0.992,0.993,0.999,0.006
heldout_qwen,0.849,0.989,0.706,0.824,0.946,0.008
reserve_human,NaN,NaN,NaN,NaN,NaN,0.007



===== MLP, preprocessing: default_no_stopwords =====
Epoch 1/20
194/194 - 7s - 34ms/step - accuracy: 0.8951 - loss: 0.2725 - val_accuracy: 0.9827 - val_loss: 0.0568
Epoch 2/20
194/194 - 1s - 5ms/step - accuracy: 0.9889 - loss: 0.0460 - val_accuracy: 0.9834 - val_loss: 0.0450
Epoch 3/20
194/194 - 1s - 6ms/step - accuracy: 0.9917 - loss: 0.0387 - val_accuracy: 0.9868 - val_loss: 0.0426


,accuracy,precision_ai,recall_ai,f1_ai,roc_auc,false_positive_rate
split,,,,,,
val,0.983,0.991,0.974,0.983,0.998,0.009
test,0.981,0.995,0.968,0.981,0.996,0.005
heldout_qwen,0.811,0.985,0.632,0.770,0.918,0.010
reserve_human,NaN,NaN,NaN,NaN,NaN,0.010


In [ ]:
# Ablation summary: the same MLP, only the preprocessing differs
summary = pd.DataFrame({v: {"val acc": r.loc["val", "accuracy"], "test acc": r.loc["test", "accuracy"],
                            "Qwen acc": r.loc["heldout_qwen", "accuracy"], "Qwen recall": r.loc["heldout_qwen", "recall_ai"],
                            "FPR reserve": r.loc["reserve_human", "false_positive_rate"]} for v, r in ablation.items()}).T
summary.to_csv(RESULTS / "ablation_preprocessing.csv")
summary

,val acc,test acc,Qwen acc,Qwen recall,FPR reserve
default,0.987,0.986,0.869,0.754,0.017
keep_case_punct,0.993,0.993,0.849,0.706,0.007
default_no_stopwords,0.983,0.981,0.811,0.632,0.010


**Choose the variant for the sequence models** by `val` accuracy (not by test or Qwen: those stay untouched for the
final comparison). The CNN and BiLSTM below use the winner.

In [ ]:
BEST_STANDARDIZE = summary["val acc"].idxmax()
print("Best preprocessing on val:", BEST_STANDARDIZE)

Best preprocessing on val: keep_case_punct


## 3. CNN

The text becomes a sequence of word ids → an embedding (each word a learned 128-dimensional vector) → 1D convolution
looking at windows of 5 words → the strongest signal of each filter (global max pooling) → Dense(64) → sigmoid.
Kryštof's CNN baseline: Embedding(20000, 128) → Conv1D(64, 5) → GlobalMaxPooling → Dense(64) → Dropout(0.3).

In [ ]:
seq_vec = layers.TextVectorization(max_tokens=MAX_TOKENS, output_mode="int", output_sequence_length=SEQ_LEN,
                                   standardize=STANDARDIZE[BEST_STANDARDIZE])
seq_vec.adapt(train_text)
seq_vectorize = lambda x, *y: (seq_vec(x), *y)

def train_sequence_model(name, body, settings, epochs=20):
    keras.utils.set_random_seed(SEED)
    inputs = keras.Input(shape=(SEQ_LEN,), dtype="int64")
    x = layers.Embedding(MAX_TOKENS, 128, mask_zero=settings.get("mask_zero", False))(inputs)
    x = body(x)
    x = layers.Dense(64, activation="relu")(x)      # head as in Kryštof's CNN and BiLSTM
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adamw", loss="binary_crossentropy", metrics=["accuracy"])
    hist = model.fit(to_dataset(splits["train"]).map(seq_vectorize),
                     validation_data=to_dataset(splits["val"]).map(seq_vectorize),
                     epochs=epochs, callbacks=[early_stop], verbose=2)
    settings = {**settings, "standardize": BEST_STANDARDIZE, "seq_len": SEQ_LEN, "embedding": 128,
                "epochs_run": len(hist.history["loss"])}
    return evaluate(name, predict_all(model, lambda x: seq_vec(x)), settings)

def cnn_body(x):
    x = layers.Conv1D(64, 5, activation="relu")(x)
    return layers.GlobalMaxPooling1D()(x)

cnn = train_sequence_model("CNN", cnn_body, {"type": "CNN", "filters": 64, "kernel": 5})
cnn

Epoch 1/20
194/194 - 14s - 74ms/step - accuracy: 0.9164 - loss: 0.2139 - val_accuracy: 0.9860 - val_loss: 0.0467
Epoch 2/20
194/194 - 1s - 4ms/step - accuracy: 0.9919 - loss: 0.0243 - val_accuracy: 0.9917 - val_loss: 0.0286
Epoch 3/20
194/194 - 1s - 4ms/step - accuracy: 0.9991 - loss: 0.0048 - val_accuracy: 0.9917 - val_loss: 0.0275


,accuracy,precision_ai,recall_ai,f1_ai,roc_auc,false_positive_rate
split,,,,,,
val,0.986,0.992,0.98,0.986,0.999,0.008
test,0.988,0.995,0.98,0.987,0.999,0.005
heldout_qwen,0.819,0.983,0.65,0.782,0.967,0.011
reserve_human,NaN,NaN,NaN,NaN,NaN,0.010


## 4. BiLSTM

The same embedding, then an LSTM that reads the review left-to-right and right-to-left and keeps a memory of what it
has read. Kryštof's BiLSTM baseline: Embedding(20000, 128) → Bidirectional(LSTM(64)) → Dense(64) → Dropout(0.3).
Our one change: `mask_zero=True`, so it ignores the padding at the end of short reviews.

In [ ]:
def bilstm_body(x):
    return layers.Bidirectional(layers.LSTM(64))(x)

bilstm = train_sequence_model("BiLSTM", bilstm_body, {"type": "BiLSTM", "units": 64, "mask_zero": True})
bilstm

Epoch 1/20
194/194 - 10s - 50ms/step - accuracy: 0.9503 - loss: 0.1390 - val_accuracy: 0.9894 - val_loss: 0.0348
Epoch 2/20
194/194 - 4s - 23ms/step - accuracy: 0.9952 - loss: 0.0182 - val_accuracy: 0.9853 - val_loss: 0.0569
Epoch 3/20
194/194 - 5s - 24ms/step - accuracy: 0.9982 - loss: 0.0069 - val_accuracy: 0.9891 - val_loss: 0.0403


,accuracy,precision_ai,recall_ai,f1_ai,roc_auc,false_positive_rate
split,,,,,,
val,0.989,0.992,0.987,0.989,0.999,0.008
test,0.987,0.989,0.985,0.987,0.999,0.011
heldout_qwen,0.784,0.980,0.579,0.728,0.943,0.012
reserve_human,NaN,NaN,NaN,NaN,NaN,0.011


## 5. All models so far (including the baselines from notebook 03)

In [ ]:
m = pd.read_csv(RESULTS / "metrics.csv")
base = Path("/content/drive/MyDrive/TA2/results/03_baselines/metrics.csv")
if base.exists():
    m = pd.concat([pd.read_csv(base), m])
table = m.pivot_table(index="model", columns="split", values="accuracy", sort=False)[["val", "test", "heldout_qwen"]]
table["Qwen recall"] = m[m.split == "heldout_qwen"].set_index("model").recall_ai
table["FPR reserve"] = m[m.split == "reserve_human"].set_index("model").false_positive_rate
table.round(3)

split,val,test,heldout_qwen,Qwen recall,FPR reserve
model,,,,,
dummy,0.500,0.500,0.500,0.000,0.000
length only,0.528,0.530,0.478,0.405,0.446
simple features,0.905,0.916,0.813,0.713,0.119
TF-IDF + logistic regression,0.990,0.990,0.828,0.671,0.011
MLP tfidf (default),0.987,0.986,0.869,0.754,0.017
MLP tfidf (keep_case_punct),0.993,0.993,0.849,0.706,0.007
MLP tfidf (default_no_stopwords),0.983,0.981,0.811,0.632,0.010
CNN,0.986,0.988,0.819,0.650,0.010
BiLSTM,0.989,0.987,0.784,0.579,0.011


Send these tables (ablation, section 5) to the team / the assistant. Then continue with notebook **05 (DistilBERT)**.